# 06. 코로나19 전후 택시 수요 변화

## 분석 배경 및 목적

COVID-19 팬데믹은 도시 교통 수요에 전례 없는 충격을 가했다. 재택근무, 사회적 거리두기, 이동 제한 조치로 택시 수요가 급감했으나, 회복 속도와 패턴은 시간대, 요일, 지역에 따라 상이하게 나타났다. 본 분석은 2018-2025년 택시 데이터를 5개 기간으로 구분하여 다음을 파악한다.

1. **수요 충격의 크기와 시점**: 코로나 초기(2020.02~) 수요 감소율과 최저점 시기
2. **시공간적 회복 패턴**: 시간대별, 요일별, 행정동별 회복 속도의 차이
3. **구조적 변화**: 코로나 이후 택시 이용 패턴이 코로나 이전과 근본적으로 달라졌는가?
4. **거리두기 단계의 영향**: 각 거리두기 단계(1~4단계)가 택시 수요에 미친 차별적 효과

**방법론적 근거**: PMC (2024)의 NYC 연구는 COVID-19가 택시 수요에 미친 시공간적 영향을 분석하며, 수요 감소가 공간적으로 균질하지 않음을 실증하였다. 상업/업무 지구의 수요 감소가 주거 지역보다 심했으며, 회복 역시 비대칭적으로 진행되었다. 본 분석은 서울 택시 데이터에 동일한 시공간 프레임워크를 적용하되, 한국 고유의 거리두기 단계 정보를 추가 변수로 활용한다.

| 구분 | 기간 |
|------|------|
| Pre-COVID | ~2020.01 |
| COVID-초기 | 2020.02~2020.12 |
| COVID-중기 | 2021 |
| COVID-후기 | 2022 |
| Post-COVID | 2023~ |

> 외부 데이터(covid_korea, social_distancing)를 조인해 확진자 수와 거리두기 단계를 함께 본다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

In [ ]:
import gc, psutil, os

def mem_usage(tag=''):
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[MEM {tag}] {gb:.2f} GB')

CHUNK_SIZE = 1_000_000
mem_usage('start')

## 1. 청크 집계: 월별 / 시간대 / 요일 / 행정동 (기간 라벨 부여)

In [ ]:
D012_PATH = './DC_TBYXD012.csv'
EXT_DIR = './external_data'
usecols = ['RIDE_DTIME','PAY_AMT','RIDE_DIST','RIDE_A_CD']
dtypes  = {'RIDE_DTIME': str,'PAY_AMT':'float64','RIDE_DIST':'float64','RIDE_A_CD': str}

def period_of(ym):
    if ym < '2020-02': return 'Pre-COVID'
    if ym <= '2020-12': return 'COVID-초기'
    if ym <= '2021-12': return 'COVID-중기'
    if ym <= '2022-12': return 'COVID-후기'
    return 'Post-COVID'

m_parts, h_parts, d_parts, a_parts = [], [], [], []
total = 0
for chunk in pd.read_csv(D012_PATH, usecols=usecols, dtype=dtypes, chunksize=CHUNK_SIZE):
    rd = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    m = rd.notna(); chunk, rd = chunk[m].copy(), rd[m]
    chunk['ym'] = rd.dt.to_period('M').astype(str)
    chunk['hour'] = rd.dt.hour
    chunk['weekday'] = rd.dt.dayofweek
    chunk['period'] = chunk['ym'].map(period_of)
    chunk['is_night'] = chunk['hour'].isin([23,0,1,2,3,4])
    total += len(chunk)
    m_parts.append(chunk.groupby('ym').agg(rides=('PAY_AMT','size'), fare_sum=('PAY_AMT','sum'),
                                            dist_sum=('RIDE_DIST','sum'), night=('is_night','sum')).reset_index())
    h_parts.append(chunk.groupby(['period','hour']).size().reset_index(name='count'))
    d_parts.append(chunk.groupby(['period','weekday']).size().reset_index(name='count'))
    a_parts.append(chunk.groupby(['period','RIDE_A_CD']).size().reset_index(name='count'))
    del chunk, rd; gc.collect()

monthly = pd.concat(m_parts).groupby('ym').sum().reset_index()
monthly['avg_fare'] = monthly['fare_sum'] / monthly['rides']
monthly['avg_dist'] = monthly['dist_sum'] / monthly['rides']
monthly['night_ratio'] = monthly['night'] / monthly['rides']
monthly['period'] = monthly['ym'].map(period_of)
monthly['date'] = pd.to_datetime(monthly['ym'])
hourly = pd.concat(h_parts).groupby(['period','hour'])['count'].sum().reset_index()
dow    = pd.concat(d_parts).groupby(['period','weekday'])['count'].sum().reset_index()
dong   = pd.concat(a_parts).groupby(['period','RIDE_A_CD'])['count'].sum().reset_index()
del m_parts, h_parts, d_parts, a_parts; gc.collect()
print(f"전체 {total:,}건"); mem_usage('after load')

## 1-2. 외부 데이터 조인 (확진자·거리두기)

In [ ]:
covid = pd.read_csv(f'{EXT_DIR}/covid_korea_2018_2026.csv', parse_dates=['date'])
dist  = pd.read_csv(f'{EXT_DIR}/social_distancing_daily.csv', parse_dates=['date'])
covid_m = covid.assign(ym=covid['date'].dt.to_period('M').astype(str)).groupby('ym')['new_cases'].sum()
dist_m  = dist.assign(ym=dist['date'].dt.to_period('M').astype(str)).groupby('ym')['distancing_level'].max()
monthly = monthly.merge(covid_m.rename('new_cases'), on='ym', how='left') \
                 .merge(dist_m.rename('distancing_level'), on='ym', how='left')
monthly[['new_cases','distancing_level']] = monthly[['new_cases','distancing_level']].fillna(0)
print('외부 데이터 조인 완료'); monthly[['ym','rides','new_cases','distancing_level']].head()

## 2. 월별 수요 시계열 + 거리두기 단계 오버레이

월별 택시 수요 시계열에 거리두기 단계를 오버레이하여 정책 개입(policy intervention)과 수요 변화의 시간적 관계를 시각화한다. 거리두기 단계 격상 시점 전후의 수요 변화를 비교하면, 정책의 인과적 효과를 추정할 수 있다. PMC (2024)의 NYC 연구에서도 이동 제한 조치(shelter-in-place order)의 시행 시점을 기준으로 택시 수요의 구조적 전환(structural break)을 분석하였다.

In [ ]:
period_colors = {'Pre-COVID':'#2196F3','COVID-초기':'#F44336','COVID-중기':'#FF9800',
                 'COVID-후기':'#FFC107','Post-COVID':'#4CAF50'}
period_order = ['Pre-COVID','COVID-초기','COVID-중기','COVID-후기','Post-COVID']
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(monthly['date'], monthly['rides'], width=25, color=monthly['period'].map(period_colors))
axt = ax.twinx()
axt.plot(monthly['date'], monthly['distancing_level'], color='black', lw=1.5, alpha=0.6, label='거리두기 단계')
axt.set_ylabel('거리두기 단계')
ax.legend(handles=[Patch(facecolor=c, label=p) for p, c in period_colors.items()], loc='upper right')
ax.set_title('월별 택시 수요 + 거리두기 단계', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 3. 시간대별 프로파일 (Pre vs 초기 vs Post)

코로나 기간별 시간대 수요 프로파일의 변화를 비교한다. 출퇴근 피크의 감소, 심야 수요의 급감, 그리고 Post-COVID에서의 프로파일 변형(재택근무 정착에 따른 출근 피크 이동 등)을 확인한다. 이 분석은 코로나가 일시적 충격이 아닌 구조적 변화를 가져왔는지를 판단하는 핵심 근거이다.

In [ ]:
mc = {p: monthly[monthly['period']==p]['ym'].nunique() for p in period_order}
fig, ax = plt.subplots(figsize=(12, 5))
for p, c in [('Pre-COVID','#2196F3'),('COVID-초기','#F44336'),('Post-COVID','#4CAF50')]:
    s = hourly[hourly['period']==p].set_index('hour')['count'] / max(mc[p],1)
    ax.plot(s.index, s.values, 'o-', color=c, lw=2, label=p)
ax.set_title('시간대별 수요 프로파일 (월평균)', fontweight='bold'); ax.set_xlabel('시간대'); ax.set_ylabel('월평균 건수')
ax.set_xticks(range(24)); ax.legend(); ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 4. 요일별 패턴 (Pre vs Post)

In [ ]:
dl = ['월','화','수','목','금','토','일']
fig, ax = plt.subplots(figsize=(10, 5))
for p, c in [('Pre-COVID','#2196F3'),('Post-COVID','#4CAF50')]:
    nm = max(mc[p], 1) * 30 / 7
    s = dow[dow['period']==p].set_index('weekday')['count'] / nm
    ax.plot(s.index, s.values, 'o-', color=c, lw=2, label=p)
ax.set_title('요일별 수요 패턴 (주평균)', fontweight='bold'); ax.set_xticks(range(7)); ax.set_xticklabels(dl)
ax.legend(); ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout(); plt.show()

## 5. 행정동별 수요 변화율 Top/Bottom 10 (Pre -> Post)

행정동별 코로나 전후 수요 변화율을 산출하여 공간적 이질성을 분석한다. PMC (2024)가 보고한 것처럼, 상업/업무 지구(강남, 여의도)는 재택근무 영향으로 수요 회복이 더딘 반면, 주거 지역은 상대적으로 빠른 회복을 보일 수 있다. 이 공간 패턴은 포스트 코로나 시대의 택시 공급 재배치 전략의 근거가 된다.

In [ ]:
pre_m, post_m = max(mc['Pre-COVID'],1), max(mc['Post-COVID'],1)
pre = dong[dong['period']=='Pre-COVID'].set_index('RIDE_A_CD')['count'] / pre_m
post = dong[dong['period']=='Post-COVID'].set_index('RIDE_A_CD')['count'] / post_m
ac = pd.DataFrame({'Pre': pre, 'Post': post}).dropna()
ac = ac[ac['Pre'] >= 10]
ac['변화율'] = (ac['Post'] - ac['Pre']) / ac['Pre'] * 100
top10, bot10 = ac.nlargest(10,'변화율'), ac.nsmallest(10,'변화율')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(top10.index.astype(str), top10['변화율'], color='#4CAF50'); axes[0].set_title('수요 증가 Top10'); axes[0].invert_yaxis()
axes[1].barh(bot10.index.astype(str), bot10['변화율'], color='#F44336'); axes[1].set_title('수요 감소 Bottom10'); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

## 6. 평균거리/요금 추이

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 5))
ax1.plot(monthly['date'], monthly['avg_dist'], color='#1976D2', lw=1.5, label='평균거리(m)')
ax1.set_ylabel('평균 이동거리(m)', color='#1976D2')
ax2 = ax1.twinx()
ax2.plot(monthly['date'], monthly['avg_fare'], color='#E64A19', lw=1.5, label='평균요금(원)')
ax2.set_ylabel('평균 요금(원)', color='#E64A19')
ax1.set_title('월별 평균거리/요금 추이', fontweight='bold')
l1,lab1 = ax1.get_legend_handles_labels(); l2,lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lab1+lab2, loc='upper left')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 7. 심야 비율 추이

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(monthly['date'], monthly['night_ratio']*100, color='#5C6BC0', lw=1.5, marker='.')
ax.set_title('월별 심야(23~04시) 비율', fontweight='bold'); ax.set_ylabel('심야 비율(%)'); ax.grid(alpha=0.3)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 8. 기간별 요약

본 분석의 핵심 발견과 실무적 시사점을 정리한다.

**실무 활용**: 코로나 영향 분석 결과는 (1) 팬데믹 유사 충격(신종 감염병, 대규모 재난)에 대한 택시 수요 시나리오 모형, (2) 포스트 코로나 시대의 택시 공급 재배치 전략, (3) 거리두기 등 정책 개입의 교통 영향 사전 평가에 활용할 수 있다. 특히 행정동별 회복 속도 차이는 지역 맞춤형 택시 정책의 근거가 된다.

In [ ]:
summary = monthly.groupby('period').agg(
    총건수=('rides','sum'), 월수=('ym','nunique'), 평균요금=('avg_fare','mean'),
    평균거리=('avg_dist','mean'), 심야비율=('night_ratio','mean')).reindex(period_order)
summary['월평균건수'] = (summary['총건수'] / summary['월수']).round(0)
base = summary.loc['Pre-COVID','월평균건수']
summary['수요변화율(%)'] = ((summary['월평균건수'] - base) / base * 100).round(1)
summary['심야비율(%)'] = (summary['심야비율']*100).round(1)
print('=== 코로나 전후 요약 ===')
summary[['총건수','월수','월평균건수','수요변화율(%)','평균요금','평균거리','심야비율(%)']]

---

## References

1. Li, M., Dong, L., Shen, Z., Lang, W., & Ye, X. (2024). Exploring spatio-temporal impact of COVID-19 on citywide taxi demand: A case study of NYC. *PLoS ONE*, 19(1), e0293362. (PMC)
2. Balbontin, C., Hensher, D. A., & Beck, M. J. (2021). Impact of COVID-19 on the number of days working from home and commuting travel. *Transport Policy*, 108, 72-85.
3. 중앙방역대책본부 (2022). 코로나19 사회적 거리두기 단계 조정 현황. 질병관리청.